# Week 6 · Wednesday: Introduction to Deep Learning

Yesterday's trees and clusters worked on columns a person chose. Today:

1. **From hand-made features to learned features:** a baseline on raw pixels
2. **One neuron:** a weighted sum, a bias and an activation
3. **Layers:** a network's forward pass, with its shapes and parameters
4. **How a CNN reads an image:** kernels, edges and pooling
5. **Text as numbers:** tokens, TF-IDF and a ticket classifier

Everything today runs in numpy, scipy and scikit-learn, so you can see every multiplication. TensorFlow arrives tomorrow.

Parts marked *Going deeper* are optional on a first pass: run their cells, then come back to them with the notes page.

Run the next cell first. It holds the imports and plotting helpers we reuse all day.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def plot_points(X, labels=None, title="", xlabel="Feature 1", ylabel="Feature 2",
                names=None, centers=None, ax=None):
    """Scatter 2D points, coloured by label if given. Label -1 is drawn as noise (DBSCAN)."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], s=45, edgecolor="black", alpha=0.85)
    else:
        for value in np.unique(labels):
            mask = labels == value
            if value == -1:
                ax.scatter(X[mask, 0], X[mask, 1], s=70, marker="x", color="black", label="Noise (-1)")
            else:
                label = names[value] if names is not None else f"{value}"
                ax.scatter(X[mask, 0], X[mask, 1], s=45, edgecolor="black", alpha=0.85, label=label)
        ax.legend()
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], s=300, marker="X", color="red",
                   edgecolor="black", linewidth=2, label="Centroids")
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if show:
        plt.show()


def plot_decision_boundary(model, X, y, title="", xlabel="Feature 1", ylabel="Feature 2",
                           names=None, ax=None):
    """Colour the background by what the model predicts at every point of a fine grid."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
    grid_predictions = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    classes = np.unique(y)
    colours = ListedColormap(plt.cm.tab10.colors[:len(classes)])   # same colours as the points
    ax.contourf(xx, yy, grid_predictions, levels=np.arange(len(classes) + 1) - 0.5, alpha=0.25, cmap=colours)
    plot_points(X, y, title, xlabel, ylabel, names=names, ax=ax)
    if show:
        plt.show()


def plot_line_chart(xs, ys, title, xlabel, ylabel):
    """A simple line chart with markers, for the elbow and silhouette plots."""
    plt.figure(figsize=(8, 5))
    plt.plot(xs, ys, marker="o", linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(xs)
    plt.grid(alpha=0.3)
    plt.show()


def show_images(images, titles):
    """Show images side by side in one row, greyscale unless the image has colour channels."""
    fig, axes = plt.subplots(1, len(images), figsize=(4.5 * len(images), 4.5))
    for ax, image, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(image, cmap="gray")
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print("Setup complete")

## 1. From hand-made features to learned features

On Monday and Tuesday a person chose the columns: temperature and humidity, income and spending score. The model learned one weight per column, and it could only be as good as those columns.

**Deep learning** is machine learning with **neural networks** that have several layers. You give the network raw input (the pixels of a photo, the words of a sentence) and the layers build their own features from it: edges first, then strokes, then whole shapes.

Before we build one, we need a **baseline**: the simplest sensible model, the score anything fancier has to beat. Our data is `load_digits`, a set of small handwritten digits that ships with scikit-learn.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print(f"Images: {digits.images.shape}   (samples, rows, columns)")
print(f"Pixel values from {digits.images.min():.0f} to {digits.images.max():.0f}")

show_images(digits.images[:6], [f"Label: {label}" for label in digits.target[:6]])

1,797 images, each an 8 by 8 grid of numbers from 0 (no ink) to 16 (full ink). To a computer, an image is only a grid of numbers.

Monday's models take one row of features per sample, so each 8 by 8 grid is laid out as a single row of 64 numbers. This is called **flattening**. scikit-learn already gives us the flattened version as `digits.data`.

In [ ]:
X_digits, y_digits = digits.data, digits.target

print(f"Flattened: {X_digits.shape}   (samples, pixels)")
print(f"Image 0 flattened equals row 0: {np.array_equal(digits.images[0].reshape(-1), X_digits[0])}")

Now the baseline: Monday's logistic regression, fed the 64 raw pixels and nothing else.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.25, random_state=RANDOM_STATE, stratify=y_digits
)

baseline = LogisticRegression(max_iter=5000)
baseline.fit(X_train, y_train)

print(f"Training accuracy: {accuracy_score(y_train, baseline.predict(X_train)):.3f}")
print(f"Test accuracy:     {accuracy_score(y_test, baseline.predict(X_test)):.3f}")

- `max_iter=5000` only lets the solver finish on these unscaled pixel values (Tuesday showed scaling). Without it, scikit-learn stops early and prints a `ConvergenceWarning`.
- A straight-line model that has never heard of a stroke or a curve scores about **0.96** on unseen digits. Small, centred images carry a lot in their raw pixels.
- The weakness is that every weight belongs to **one pixel position**. The model has no idea that pixel 10 and pixel 11 are neighbours, and a digit drawn one pixel to the right looks like different data to it. Learned features fix that, and section 4 shows how.

Try changing `test_size=0.25` to `test_size=0.5` in the split. The model trains on fewer digits; the test score moves by about a point and stays far above chance. Change it back.

## 2. One neuron

A network is made of **neurons**. One neuron does three things:

1. Multiplies each input by its own **weight** and adds the results up.
2. Adds one more number, the **bias**, which shifts the result up or down.
3. Passes the total through an **activation**, a fixed function that decides what comes out.

```
z      = x1*w1 + x2*w2 + x3*w3 + b
output = activation(z)
```

Two activations for today, one line each (Friday is their day):

- **ReLU** keeps a positive number and turns a negative one into 0.
- **Sigmoid** squashes any number into the range 0 to 1.

Before running the cell, work it out on paper: inputs 2, 1, 3; weights 0.5, -1, 0.25; bias 0.5. What is `z`, and what does ReLU give?

In [ ]:
x = np.array([2.0, 1.0, 3.0])       # three inputs
w = np.array([0.5, -1.0, 0.25])     # one weight per input
b = 0.5                             # the bias

z = x @ w + b                       # weighted sum plus bias
relu = np.maximum(0, z)             # ReLU: negatives become 0
sigmoid = 1 / (1 + np.exp(-z))      # sigmoid: any number into 0 to 1

print(f"z = {z}")
print(f"ReLU output:    {relu}")
print(f"Sigmoid output: {sigmoid:.3f}")

- `x @ w` multiplies the matching positions and adds them up, which is called the **dot product**: `2*0.5 + 1*(-1) + 3*0.25 = 0.75`. Adding the bias gives `z = 1.25`.
- ReLU passes 1.25 through unchanged, because it is positive. Sigmoid turns it into 0.777.
- The weights and the bias are the neuron's **parameters**, the numbers training would adjust. Here we chose them by hand.

Try changing the bias to `b = -2.0`. Now `z` is -1.25, ReLU outputs 0.0 and sigmoid about 0.223.

## 3. Layers: a network's forward pass

A **layer** is many neurons reading the same inputs, each with its own weights. In numpy we store a layer as:

- a **weight matrix** of shape `(inputs, units)`: one column of weights per neuron (a neuron is also called a **unit**)
- a **bias vector** of shape `(units,)`: one bias per neuron

Multiplying the inputs by the weight matrix works **rows times columns**: each sample (a row of inputs) meets each neuron (a column of weights) in one dot product. Here are 2 samples going into a layer of 2 neurons. Sample 1 and neuron 1 are exactly the neuron from section 2.

In [ ]:
X_small = np.array([[2.0, 1.0, 3.0],
                    [0.0, 4.0, 1.0]])      # 2 samples, 3 inputs each

W_small = np.array([[0.5, 1.0],
                    [-1.0, 0.0],
                    [0.25, 2.0]])          # 3 inputs, 2 neurons: one column each
b_small = np.array([0.5, -1.0])            # one bias per neuron

print(f"{X_small.shape} @ {W_small.shape} -> {(X_small @ W_small + b_small).shape}")
print(X_small @ W_small + b_small)         # row = sample, column = neuron

- Top-left is 1.25, our neuron from section 2. Each row is one sample, each column one neuron.
- The inner numbers of the shapes have to match: `(2, 3) @ (3, 2)`. The 3 inputs of each sample meet 3 weights per neuron. If they don't match, numpy refuses to multiply.

Now a real network for the digits, **64 → 16 → 10**:

- the **input layer** is the 64 pixels
- a **hidden layer** of 16 neurons with ReLU (hidden means it sits between input and output)
- an **output layer** of 10 neurons, one score per digit 0 to 9; the raw scores are usually called **logits**

The weights are **random**. Running an input through the layers in order is a **forward pass**: input in, answer out, no learning.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
hidden_size = 16                                    # neurons in the hidden layer

W1 = rng.normal(0, 0.1, size=(64, hidden_size))     # (inputs, units)
b1 = np.zeros(hidden_size)
W2 = rng.normal(0, 0.1, size=(hidden_size, 10))     # 10 outputs, one per digit
b2 = np.zeros(10)

X_batch = X_test[:5]                                # five test digits
h = np.maximum(0, X_batch @ W1 + b1)                # hidden layer, ReLU
logits = h @ W2 + b2                                # output layer, one score per digit

print(f"Input:  {X_batch.shape}")
print(f"Hidden: {h.shape}")
print(f"Output: {logits.shape}")

Five digits went in as `(5, 64)`, became `(5, 16)` after the hidden layer and `(5, 10)` at the output: ten scores per digit.

How many numbers does this network hold? Each layer has `inputs × units` weights plus `units` biases.

In [ ]:
layer_1 = W1.size + b1.size
layer_2 = W2.size + b2.size

print(f"Layer 1: 64 x {hidden_size} + {hidden_size} = {layer_1:,}")
print(f"Layer 2: {hidden_size} x 10 + 10 = {layer_2:,}")
print(f"Total parameters: {layer_1 + layer_2:,}")

1,210 parameters. How often will this network get the digit right? Its guess is the output with the highest score. `argmax(axis=1)` gives the position of the largest value in each row, and the position is the digit.

In [ ]:
h_all = np.maximum(0, X_test @ W1 + b1)
guesses = (h_all @ W2 + b2).argmax(axis=1)     # the highest of the ten scores

print(f"Guesses:     {guesses[:10]}")
print(f"True labels: {y_test[:10]}")
print(f"Accuracy:    {accuracy_score(y_test, guesses):.3f}")

- Accuracy is about **0.09**, close to one in ten: chance for ten classes. Random weights carry no information, so the network keeps guessing the same few digits.
- The structure is all there: layers, shapes, 1,210 parameters. What is missing is **training**, adjusting those 1,210 numbers until the guesses are right. That is week 7 (backpropagation). Friday trains a single neuron.
- `rng = np.random.default_rng(RANDOM_STATE)` fixes the random weights, so everyone sees the same numbers.

Try changing `hidden_size = 16` to `32` in the network cell, then run the three cells again. The hidden shape becomes `(5, 32)`, the count becomes 2,410 (64 × 32 + 32 = 2,080, plus 32 × 10 + 10 = 330), and accuracy stays useless: it prints 0.011, below chance this time. Twice the neurons with random weights is still a guess. Change it back.

## 4. How a CNN reads an image

A **convolutional neural network (CNN)** is the kind of network built for images. Instead of one weight per pixel, it slides small grids of weights across the image.

### 4.1 A kernel slides over an image

A **kernel** (also called a filter) is a small grid of weights. This one is the **vertical-edge kernel**:

```
-1  0  1
-1  0  1
-1  0  1
```

Put it on a 3 by 3 patch of the image, multiply the matching numbers and add them up: the same weighted sum as one neuron. Then slide it one step and do it again, everywhere it fits. The grid of results is a **feature map**. This kernel gives a big positive number where the image goes from dark on the left to bright on the right: a vertical edge.

We slide it with `scipy.signal.correlate2d`, which is exactly what a CNN's `Conv2D` layer computes. Deep learning calls this **convolution**, but strictly it is **cross-correlation**: the kernel slides unflipped. `scipy.signal.convolve2d` flips the kernel first and would light up the opposite edge, so we don't use it.

In [ ]:
from scipy.signal import correlate2d

toy = np.array([[0, 0, 9, 9],
                [0, 0, 9, 9],
                [0, 0, 9, 9],
                [0, 0, 9, 9]])           # dark on the left, bright on the right

kernel = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]])          # the vertical-edge kernel

print("Kernel slid as it is (correlate2d, what Conv2D does):")
print(correlate2d(toy, kernel, mode="valid"))
print("Kernel flipped first (what convolve2d would do):")
print(correlate2d(toy, np.flip(kernel), mode="valid"))

- Every position covers the dark-to-bright step, so each result is `-(0+0+0) + (9+9+9) = 27`. The flipped kernel reads the same edge backwards: -27.
- `mode="valid"` only places the kernel where it fits completely inside the image. A 3 by 3 kernel fits in `4 - 3 + 1 = 2` positions each way, so a 4 by 4 image gives a 2 by 2 map.

Now the same kernel over a real digit, a zero. Before running: a 3 by 3 kernel on an 8 by 8 digit, what shape comes out?

In [ ]:
digit = digits.images[0]                     # the first digit, a 0
edge_kernel = kernel
digit_edges = correlate2d(digit, edge_kernel, mode="valid")

print(digit.astype(int))
print(f"{digit.shape} -> {digit_edges.shape}")
print(digit_edges.astype(int))

show_images([digit, digit_edges], ["Digit (label 0)", "Feature map"])

- 6 by 6: the kernel fits in `8 - 3 + 1 = 6` positions across and 6 down.
- In the feature map, white is a large positive number: the left side of a stroke, where ink starts as you move right (33 and 40 in the first column). Black is a large negative number: the right side of a stroke, where ink stops (-34 in the last column).
- Grey is near zero: flat areas with no vertical edge.

The same kernel works on any image size. Here it is on a photo that ships with scikit-learn. The photo has three colour channels (red, green, blue), so we average them into one greyscale grid first.

In [ ]:
from sklearn.datasets import load_sample_image

china = load_sample_image("china.jpg")
grey = china.mean(axis=2)                    # average red, green and blue
photo_edges = correlate2d(grey, edge_kernel, mode="valid")

print(f"Photo {china.shape} {china.dtype} -> greyscale {grey.shape} -> feature map {photo_edges.shape}")

show_images([china, grey, np.abs(photo_edges)], ["Photo", "Greyscale", "Edge strength"])

- The photo is `(427, 640, 3)`: 427 rows, 640 columns, 3 colour values per pixel. `uint8` means whole numbers from 0 to 255.
- After the kernel it is `(425, 638)`: two rows and two columns smaller, as with the digit.
- `np.abs` shows edges in either direction as white. The pagoda's pillars and the sides of each storey light up; the flat sky stays black.

Try changing `edge_kernel = kernel` to `edge_kernel = kernel.T` (the **transpose**, the kernel turned on its side, a horizontal-edge kernel) in the digit cell, then run both cells again. The top and bottom strokes of the zero light up, and in the photo the roof edges and balcony rails take over from the pillars. Change it back.

### 4.2 ReLU, then max pooling

A CNN layer passes its feature map through ReLU, so only the edges the kernel was looking for survive (negatives become 0).

Then comes **max pooling**: cut the map into 2 by 2 windows and keep only the largest number in each. The map shrinks by half each way.

In [ ]:
def max_pool(feature_map, size=2):
    """Keep the largest value in each size-by-size window; the windows do not overlap."""
    rows, cols = feature_map.shape[0] // size, feature_map.shape[1] // size
    pooled = np.zeros((rows, cols))
    for r in range(rows):
        for c in range(cols):
            window = feature_map[r * size:(r + 1) * size, c * size:(c + 1) * size]
            pooled[r, c] = window.max()
    return pooled


digit_map = np.maximum(0, digit_edges)       # ReLU
digit_pooled = max_pool(digit_map, size=2)

print(f"{digit_map.shape} -> {digit_pooled.shape}")
print(digit_pooled.astype(int))
show_images([digit_edges, digit_map, digit_pooled], ["Feature map", "After ReLU", "After 2 by 2 max pooling"])

- The 6 by 6 map becomes 3 by 3. The top-left window held 33, 27, 40 and 10, so it keeps **40**.
- Pooling throws away exactly where inside the window the edge was. In return the next layer has a quarter of the numbers to read, and a stroke that moves by one pixel often lands in the same window: the network copes with small shifts.

Try changing `size=2` to `size=3` in the `max_pool(digit_map, ...)` line. The 6 by 6 map shrinks to 2 by 2, keeping even less position. Change it back.

The photo goes through the same two steps:

In [ ]:
photo_map = np.maximum(0, photo_edges)       # ReLU
photo_pooled = max_pool(photo_map, size=2)

print(f"{photo_map.shape} -> {photo_pooled.shape}")
show_images([photo_map, photo_pooled], ["After ReLU", "After 2 by 2 max pooling"])

`(425, 638)` becomes `(212, 319)`: half each way, with the odd last row and column dropped. The pooled picture still shows the same edges with a quarter of the numbers.

### 4.3 A CNN's stack

A real CNN repeats these steps, then hands the result to the kind of layers we built in section 3:

```
image -> [convolution -> ReLU -> max pooling] -> [convolution -> ReLU -> max pooling] -> flatten -> dense layer -> output (one score per class)
```

- A **dense layer** is the section 3 kind: every input connected to every neuron. **Flatten** is what we did to the digits in section 1.
- The big difference from today: a CNN **learns its kernel numbers** in training. We wrote one edge kernel by hand. A CNN learns dozens per layer, and finds patterns nobody would think to draw. Early layers end up finding edges; later layers combine them into strokes and shapes.
- In Keras, the library we start tomorrow, these steps are the layers `Conv2D`, `MaxPooling2D`, `Flatten` and `Dense`.

Why not flatten the photo and use a dense layer straight away? Count the parameters.

In [ ]:
height, width = 427, 640                   # the photo's size
hidden_units = 16

dense_params = height * width * hidden_units + hidden_units   # one weight per pixel per neuron, plus biases
kernel_params = 3 * 3 + 1                                     # nine weights and one bias, reused at every position

print(f"Dense layer of {hidden_units} neurons on {height} x {width} pixels: {dense_params:,} parameters")
print(f"One 3 x 3 kernel on the same image:          {kernel_params} parameters")

- A dense layer of only 16 neurons on the flattened photo needs **4,372,496** parameters, and each weight is tied to one pixel position.
- A kernel needs **10**, whatever the image size, because the same 9 weights and 1 bias are reused at every position. It finds its pattern anywhere in the image. That is why CNNs, not flat classifiers, read large images.

Try changing `height, width = 427, 640` to `8, 8`, the digit's size. The dense count drops to 1,040 and the kernel stays at 10.

## 5. Text as numbers

**Natural language processing (NLP)** is getting computers to work with human language: filtering spam, reading sentiment, routing support tickets, translating, answering in a chat. Every model we've seen reads only numbers, so the first job is always the same: turn text into numbers.

### 5.1 Tokens and a vocabulary

Our data is 24 support-ticket titles, each labelled `billing` or `technical`. Six titles come from the ticket desk's seed data in week 4 (such as "Login page returns 500"). The other 18 titles, and every label, are new: we made them up for this notebook (the seed data has no such label).

In [ ]:
tickets = [
    ("Login page returns 500", "technical"), ("Invoice PDF is blank", "billing"),
    ("Typo on pricing page", "technical"), ("Checkout button frozen", "technical"),
    ("Export is missing rows", "technical"), ("Add dark mode", "technical"),
    ("Charged twice this month", "billing"), ("Refund not received yet", "billing"),
    ("Update credit card on file", "billing"), ("Invoice shows wrong amount", "billing"),
    ("Cancel my subscription", "billing"), ("Need a receipt for March", "billing"),
    ("Question about my plan price", "billing"), ("Discount code not applied", "billing"),
    ("Payment declined at renewal", "billing"), ("Tax missing from invoice", "billing"),
    ("Change billing address", "billing"), ("App crashes on startup", "technical"),
    ("Password reset email never arrives", "technical"), ("Search returns no results", "technical"),
    ("Dashboard loads very slowly", "technical"), ("Error 404 on settings page", "technical"),
    ("Cannot upload profile photo", "technical"), ("Sync fails after update", "technical"),
]
texts = [title for title, label in tickets]
labels = [label for title, label in tickets]

print(f"{len(texts)} titles: {labels.count('billing')} billing, {labels.count('technical')} technical")

Three steps turn a title into numbers:

- **Tokens:** split the text into words. `CountVectorizer` lowercases everything and keeps runs of two or more letters or digits, so "500" is a token and a lone "a" is dropped.
- **Vocabulary:** every distinct token becomes a column.
- **Counts:** each title becomes a row saying how many times each vocabulary word appears.

Counting words and ignoring their order is called **bag of words**. Here it is on three of the seed titles:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

titles = ["Login page returns 500", "Typo on pricing page", "Invoice PDF is blank"]

counter = CountVectorizer()
counts = counter.fit_transform(titles)       # learn the vocabulary, then count

print(f"Vocabulary: {counter.get_feature_names_out()}")
print(counts.toarray())                      # row = title, column = vocabulary word

- 11 distinct words, so 11 columns, in alphabetical order. Row 1 has a 1 under '500', 'login', 'page' and 'returns'.
- 'page' is the only word in two titles: its column has two 1s.
- `fit_transform` does two jobs: `fit` learns the vocabulary, `transform` counts. The counts come back in a compact **sparse** form that stores only the non-zero numbers; `toarray()` shows the full table.

Try changing `CountVectorizer()` to `CountVectorizer(stop_words="english")`. Common words such as "is" and "on" (**stop words**) leave the vocabulary. Change it back.

### 5.2 TF-IDF and n-grams

*Going deeper. The one idea to keep: TF-IDF scores a word higher when it is rare across the titles, and an n-gram keeps neighbouring words together.*

Raw counts treat every word the same. **TF-IDF** (term frequency, inverse document frequency) scores a word higher when it appears often in **this** title and lower when it appears in many titles. Words that every ticket shares tell you little about any one ticket. Each row is also scaled so that long and short titles compare fairly.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
weights = tfidf.fit_transform(titles)

print(f"Vocabulary: {tfidf.get_feature_names_out()}")
print(weights.toarray().round(2))

- In the first title, 'page' scores **0.4** while 'login', 'returns' and '500' score **0.53**: 'page' also appears in the second title, so it counts for less.
- The third title shares no word with the others, so its four words score the same, **0.5** each.

Bag of words has a blind spot: it **ignores word order**. These two tickets mean opposite things:

In [ ]:
opposite = ["Refund blocked, login works", "Login blocked, refund works"]

bag = CountVectorizer()
rows = bag.fit_transform(opposite)

print(f"Vocabulary: {bag.get_feature_names_out()}")
print(rows.toarray())

Identical rows: both titles hold the same four words once each. An **n-gram** is a run of n neighbouring words kept together as one vocabulary entry; a **bigram** is two. `ngram_range=(1, 2)` keeps single words **and** neighbouring pairs.

In [ ]:
pairs = TfidfVectorizer(ngram_range=(1, 2))      # single words and neighbouring word pairs

pairs.fit(["Login page returns 500"])
print(f"One title: {pairs.get_feature_names_out()}")

pair_rows = pairs.fit_transform(opposite)
print(f"Two opposite tickets: {pairs.get_feature_names_out()}")
print(pair_rows.toarray().round(2))

- One title now gives 7 entries: its 4 words plus 'login page', 'page returns' and 'returns 500'.
- The two opposite tickets now differ: 'refund blocked' belongs to the first, 'login blocked' to the second. Bigrams bring back some of the order.
- The price is a bigger vocabulary: more columns for the same text.

Try changing `ngram_range=(1, 2)` to `ngram_range=(1, 1)`. The pairs disappear and the two ticket rows are identical again. Change it back.

### 5.3 A classifier on ticket titles

Now the whole thing as a **pipeline**, the shape of every text classifier:

```
tokenizer -> vocabulary -> vectorizer -> model
(split)      (columns)     (TF-IDF)      (logistic regression)
```

`make_pipeline` chains the vectorizer and the model, so `fit` and `predict` take plain strings. `max_iter=5000` is the same line as in section 1: it lets the solver finish.

In [ ]:
from sklearn.pipeline import make_pipeline

classifier = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)),
                           LogisticRegression(max_iter=5000))
classifier.fit(texts, labels)

new = ["Refund for a double charge", "Settings page crashes", "Blank invoice"]
probabilities = classifier.predict_proba(new)

print(f"Classes: {classifier.classes_}")
for title, label, row in zip(new, classifier.predict(new), probabilities):
    print(f"{title:<28} -> {label:<10} {row.round(2)}")

- All three new titles get the label a person would give. "Refund", "blank" and "invoice" appear in billing titles; "page" and "crashes" in technical ones.
- "charge" counts for nothing: the training title says "Charged", a different token. The model only knows the exact words it was trained on.
- The probabilities (billing first, then technical) sit near 0.5 to 0.6: with only 24 examples the model is never very sure.
- **24 titles with labels we invented is a toy.** It shows the pipeline; it proves nothing about how well ticket routing works.

What happens to a title made only of words the model has never seen?

In [ ]:
vectorizer = classifier.named_steps["tfidfvectorizer"]
unseen = ["Quantum flux capacitor"]

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Non-zero numbers for the unseen title: {vectorizer.transform(unseen).nnz}")
print(f"Probabilities: {classifier.predict_proba(unseen).round(2)}")

- The vocabulary was fixed when we called `fit`: 155 words and word pairs. New text goes through `transform`, which uses that vocabulary and silently drops any word not in it.
- All three words are unknown, so the title becomes a row of zeros (0 non-zero numbers) and the model can only say 50/50.
- Never call `fit_transform` on a new title: it would build a fresh vocabulary from that one title, and the columns would no longer match what the model learned. The pipeline's `predict` calls `transform` for you.

Try changing the `new` list in the classifier cell to your own titles: first a billing one such as "Refund for a duplicate invoice" (billing, 0.59), then "Purple elephants dancing" (0.5 and 0.5: no known words, and the printed label is only the tie going to the first class). Change it back.

Real NLP systems (translation, chat assistants) use much bigger networks and learn a list of numbers per token instead of counting words, but they start the same way: text turned into numbers.

## Recap

| Idea | Type | Good at | Watch out for |
|---|---|---|---|
| Logistic regression on raw pixels | classical baseline | a quick score to beat on small, centred images | one weight per pixel position; needs `max_iter=5000` on unscaled pixels |
| Neuron | building block | weighted sum + bias, then an activation | ReLU zeroes negatives; sigmoid squashes into 0 to 1 |
| Dense layers, forward pass | deep learning | layers that build their own features | shapes are `(inputs, units)` and must line up; random weights guess about 1 in 10 |
| Kernel (convolution) | CNN layer | finds a pattern anywhere, 10 parameters for 3 by 3 | use `correlate2d`, not `convolve2d`; a CNN learns its kernels |
| Max pooling | CNN layer | shrinks the map, copes with small shifts | throws away the exact position |
| Bag of words, TF-IDF, n-grams | NLP technique | text to numbers, fast and simple | bag of words ignores order; unseen words are dropped |
| TF-IDF + logistic regression pipeline | NLP classifier | ticket routing, spam filtering | 24 invented labels is a toy; never `fit_transform` new text |

**Quick check**
1. A neuron has inputs 2, 1 and 3, weights 0.5, -1 and 0.25, and a bias of -2. What does ReLU output?
2. How many parameters does the 64-16-10 network have, and how did you count them?
3. A 3 by 3 kernel slides over an 8 by 8 digit with no padding. What shape comes out? And after 2 by 2 max pooling?
4. Why do we run the kernel with `correlate2d` and not `convolve2d`?
5. Why do "Refund blocked, login works" and "Login blocked, refund works" get the same bag-of-words row, and what brings back some of the order?
6. A teammate calls `fit_transform` on a new ticket title before predicting. What goes wrong?